# Week 5: TF-IDF and Logistic Regression Baseline

This notebook builds the first working TF-IDF + Logistic Regression baseline model for financial complaint product classification.

Scope rules:

- Use only `data/processed/cfpb_complaints_2024_cleaned.csv`.
- Do not use the 2025 holdout dataset.
- Do not include raw complaint narrative examples in notebook outputs.
- Do not create confusion matrix visuals or routing confidence outputs.
- Do not save model artifacts.
- Treat this as a first baseline, not final model selection.

## 1. Imports and Project Paths

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [2]:
def find_project_root(start_path):
    """Find the project root from either the repository root or the notebooks folder."""
    current_path = start_path.resolve()
    for candidate_path in [current_path, *current_path.parents]:
        expected_data_path = candidate_path / "data" / "processed" / "cfpb_complaints_2024_cleaned.csv"
        if expected_data_path.exists():
            return candidate_path
    raise FileNotFoundError("Could not find data/processed/cfpb_complaints_2024_cleaned.csv")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cfpb_complaints_2024_cleaned.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATA_PATH.relative_to(PROJECT_ROOT)}")

Project root: E:\MGA\ITEC6740\Final-Project\financial-complaint-auto-routing-nlp
Dataset path: data\processed\cfpb_complaints_2024_cleaned.csv


## 2. Load Cleaned 2024 Modeling Dataset

In [3]:
df = pd.read_csv(DATA_PATH)

print(f"Loaded rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")

Loaded rows: 50,000
Columns: ['clean_complaint_text', 'product']


In [4]:
required_columns = ["clean_complaint_text", "product"]
missing_required_columns = [column for column in required_columns if column not in df.columns]

if missing_required_columns:
    raise ValueError(f"Missing required columns: {missing_required_columns}")

print(f"Required columns present: {required_columns}")

Required columns present: ['clean_complaint_text', 'product']


## 3. Modeling Scope and Class Distribution

In [5]:
X = df["clean_complaint_text"].fillna("").astype(str)
y = df["product"]

class_counts = y.value_counts().rename_axis("product").reset_index(name="count")
minimum_class_count = 500

selected_class_counts = class_counts[class_counts["count"] >= minimum_class_count].copy()
excluded_class_counts = class_counts[class_counts["count"] < minimum_class_count].copy()
selected_classes = selected_class_counts["product"].tolist()

print("Week 5 Version 1 baseline scope: classes with at least 500 complaints.")
print("This is the first baseline modeling scope, not final model selection.")
print(f"Selected classes: {len(selected_classes)}")
print(selected_class_counts.to_string(index=False))

print("\nExcluded low-count classes for this first baseline scope:")
print(excluded_class_counts.to_string(index=False))

Week 5 Version 1 baseline scope: classes with at least 500 complaints.
This is the first baseline modeling scope, not final model selection.
Selected classes: 8
                                            product  count
Credit reporting or other personal consumer reports  36128
                                    Debt collection   5357
                                        Credit card   2735
                        Checking or savings account   2146
                                           Mortgage    887
 Money transfer, virtual currency, or money service    725
                                       Student loan    628
                              Vehicle loan or lease    590

Excluded low-count classes for this first baseline scope:
                                                product  count
Payday loan, title loan, personal loan, or advance loan    363
                                           Prepaid card    317
                              Debt or credit management    1

In [6]:
modeling_df = df[df["product"].isin(selected_classes)].copy()

X_model = modeling_df["clean_complaint_text"].fillna("").astype(str)
y_model = modeling_df["product"]

print(f"Rows before filtering: {len(df):,}")
print(f"Rows used for Week 5 baseline: {len(modeling_df):,}")
print(f"Rows excluded by low-count class filter: {len(df) - len(modeling_df):,}")
print(f"Product classes used: {y_model.nunique():,}")

Rows before filtering: 50,000
Rows used for Week 5 baseline: 49,196
Rows excluded by low-count class filter: 804
Product classes used: 8


## 4. Train/Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.20,
    random_state=42,
    stratify=y_model,
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print("Stratified split used to preserve product category distribution.")

Training rows: 39,356
Test rows: 9,840
Stratified split used to preserve product category distribution.


## 5. TF-IDF + Logistic Regression Pipeline

In [8]:
baseline_pipeline = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                max_features=50000,
                ngram_range=(1, 2),
                min_df=2,
            ),
        ),
        (
            "logistic_regression",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

print("Pipeline summary:")
print("- TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2)")
print('- LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)')
print("This is a first baseline model, not a final model selection.")

Pipeline summary:
- TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2)
- LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
This is a first baseline model, not a final model selection.


## 6. Baseline Training

In [9]:
baseline_pipeline.fit(X_train, y_train)

print("Baseline model training complete.")

Baseline model training complete.


## 7. Test Set Evaluation

In [10]:
y_pred = baseline_pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_test,
    y_pred,
    average="macro",
    zero_division=0,
)
weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0,
)

metrics_table = pd.DataFrame(
    {
        "metric": [
            "accuracy",
            "macro_precision",
            "macro_recall",
            "macro_f1",
            "weighted_precision",
            "weighted_recall",
            "weighted_f1",
        ],
        "value": [
            accuracy,
            macro_precision,
            macro_recall,
            macro_f1,
            weighted_precision,
            weighted_recall,
            weighted_f1,
        ],
    }
)
metrics_table["value"] = metrics_table["value"].round(4)

print(metrics_table.to_string(index=False))

            metric  value
          accuracy 0.8622
   macro_precision 0.6803
      macro_recall 0.8080
          macro_f1 0.7332
weighted_precision 0.8885
   weighted_recall 0.8622
       weighted_f1 0.8703


In [11]:
classification_metrics = classification_report(
    y_test,
    y_pred,
    labels=selected_classes,
    output_dict=True,
    zero_division=0,
)

per_class_metrics = (
    pd.DataFrame(classification_metrics)
    .transpose()
    .loc[selected_classes, ["precision", "recall", "f1-score", "support"]]
    .rename(columns={"f1-score": "f1"})
)
per_class_metrics[["precision", "recall", "f1"]] = per_class_metrics[["precision", "recall", "f1"]].round(4)
per_class_metrics["support"] = per_class_metrics["support"].astype(int)

print(per_class_metrics.to_string())

                                                     precision  recall      f1  support
Credit reporting or other personal consumer reports     0.9754  0.8779  0.9241     7226
Debt collection                                         0.6380  0.8451  0.7271     1072
Credit card                                             0.6065  0.8227  0.6982      547
Checking or savings account                             0.7445  0.7879  0.7656      429
Mortgage                                                0.7624  0.8701  0.8127      177
Money transfer, virtual currency, or money service      0.6306  0.6828  0.6556      145
Student loan                                            0.6467  0.8571  0.7372      126
Vehicle loan or lease                                   0.4381  0.7203  0.5449      118


## 8. Week 5 Summary

The Week 5 baseline uses the cleaned 2024 modeling dataset and keeps product classes with at least 500 complaints. This follows the Week 4 recommendation to start Version 1 with stronger product categories because the full product distribution is highly imbalanced.

The model is a simple Scikit-learn pipeline with TF-IDF text features and Logistic Regression. It is the first working baseline only, not final model selection. No 2025 holdout data is used, and the 2025 dataset remains separate for future out-of-time validation.

In [12]:
print("Safety reminder:")
print("Safe Week 5 files to stage should be only:")
print("- notebooks/04_sklearn_baseline_model.ipynb")
print("- reports/results_summary.md")
print("Do not stage CSV files.")
print("Do not stage model artifacts.")
print("Do not stage figures for Week 5.")

Safety reminder:
Safe Week 5 files to stage should be only:
- notebooks/04_sklearn_baseline_model.ipynb
- reports/results_summary.md
Do not stage CSV files.
Do not stage model artifacts.
Do not stage figures for Week 5.
